# Pipeline FOL (đúng thứ tự bạn yêu cầu)

1. **Dataset** + in vài `eval_prompt` (train).
2. **Trước SFT:** base model — **10 greedy ngẫu nhiên (train)** + **full test** exact-match + NLL → `fol_preflight_baseline.json`.
3. **SFT LoRA** → eval greedy train/dev/test (theo `FOL_EVAL_FOL_MAX_SAMPLES`) + benchmark test → `fol_test_benchmark.json` / `experiment_log.json`.
4. **Push Hub** (kèm các JSON metrics).
5. **Tải merged từ Hub** + metrics + **20 mẫu test ngẫu nhiên**.

**VRAM:** sau Phase 6 baseline, `del`/GC trước Phase 7 train; sau train lại GC trước Hub reload.

**Thứ tự môi trường:** Phase 1 (nạp `.env` / token) → **Phase 1b `%pip install unsloth`** nếu bật Unsloth → ô **Chọn `RUNTIME`** → …


## Phase 1 — Cài đặt


In [29]:
# %pip install trl python-dotenv
# %pip install peft
# %pip install unsloth
# %pip install -U bitsandbytes
# %pip uninstall -y torch torchvision torchaudio
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !python3 -m pip uninstall -y transformers
# !python3 -m pip install --upgrade "transformers>=4.52.0"

In [44]:
# Import unsloth trước trl (patch TRL / SFTConfig). Bỏ qua nếu chưa cài unsloth.
try:
    from unsloth import FastLanguageModel  # noqa: F401
except ImportError:
    try:
        import unsloth  # noqa: F401
    except ImportError:
        pass

In [45]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=False)
GITHUB_TOKEN = (os.environ.get("GIT_TOKEN") or "").strip()
HF_TOKEN = (os.environ.get("HF_TOKEN") or "").strip()
if GITHUB_TOKEN and HF_TOKEN:
    print('Load env variable successfully')

Load env variable successfully


## Phase 1b — Gói `unsloth` (khi `use_unsloth=True`)

Nếu train báo `No module named 'unsloth'` hoặc “pip cài vào Python khác kernel”: chạy **ô code ngay dưới** (`%pip` luôn gắn với **Python của kernel** — thường là `/venv/main/bin/python`). Xong nên **Kernel → Restart**, rồi chạy lại từ đầu Phase 1.

Nếu không dùng Unsloth: trong `.env` hoặc YAML đặt `FOL_USE_UNSLOTH=false` / `use_unsloth: false`.

**Thứ tự import:** `train.py` đã nạp `unsloth` (nếu có) *trước* `trl`. Ô code `FastLanguageModel` / `unsloth` phía trên chỉ để nhắc thứ tự khi bạn tự thêm `trl` trong notebook — có thể bỏ qua nếu không cài Unsloth.


In [32]:
# import sys

# print("Kernel Python (phải trùng nơi pip cài):", sys.executable)
# # Cài vào đúng interpreter của notebook — sau đó Restart kernel nếu Jupyter báo cần
# %pip install -q "unsloth"


## Chọn môi trường: `local` \| `colab` \| `kaggle`

Chạy **ô code ngay dưới** (hoặc để `RUNTIME = None` để **tự nhận**: Kaggle → Colab → local).

| Giá trị | Ý nghĩa |
|---------|--------|
| `local` | Không clone; repo đã có trên máy — mở notebook từ `.../Logic_Based_Educational_Queries_Project/notebooks` hoặc set `LOGIC_PROJECT_ROOT`. |
| `colab` | Clone vào `/content/project`, checkout nhánh, set `LOGIC_PROJECT_ROOT`. |
| `kaggle` | Clone vào `/kaggle/working/project`, checkout nhánh, set `LOGIC_PROJECT_ROOT`. |

Có thể export `FOL_RUNTIME` hoặc `LOGIC_RUNTIME` (ví dụ `local`) thay vì sửa ô dưới.


In [33]:
# Một trong: "local" | "colab" | "kaggle" — hoặc None = tự nhận diện
RUNTIME = 'local'  # ví dụ local: RUNTIME = "local"

import os
from pathlib import Path


def _fol_detect_runtime() -> str:
    if Path("/kaggle").is_dir():
        return "kaggle"
    if Path("/content").is_dir():
        return "colab"
    return "local"


_rt = RUNTIME
if _rt is None or (isinstance(_rt, str) and not str(_rt).strip()):
    _rt = (
        os.environ.get("FOL_RUNTIME", "").strip().lower()
        or os.environ.get("LOGIC_RUNTIME", "").strip().lower()
        or None
    )
if _rt is None:
    RUNTIME = _fol_detect_runtime()
else:
    RUNTIME = str(_rt).strip().lower()
    if RUNTIME not in ("local", "colab", "kaggle"):
        raise ValueError("RUNTIME phải là 'local', 'colab', 'kaggle' hoặc None (tự nhận)")

print("RUNTIME =", RUNTIME)


RUNTIME = local


## (Tuỳ chọn) Clone trên Colab / Kaggle
Nếu `RUNTIME=local` thì **bỏ qua** ô clone bên dưới. Nếu `colab` / `kaggle`: chạy ô clone rồi **Phase 3 — Bootstrap**.


## Clone repo + checkout nhánh *(chỉ khi `RUNTIME` là `colab` hoặc `kaggle`)*

**Hai chỗ token (khác nhau):**

1. **GitHub (ô code dưới)** — `TOKEN` classic PAT (repo private) hoặc Secret/env `GITHUB_TOKEN` / repo public.
2. **Hugging Face (Phase 2)** — Secret `HF_Token` (Kaggle) hoặc `HF_TOKEN` / `.env` (local, Colab).

Sửa `GITHUB_OWNER_REPO`, `GIT_BRANCH` nếu cần.


In [34]:
import os
import subprocess
from pathlib import Path


def _fol_detect_runtime() -> str:
    if Path("/kaggle").is_dir():
        return "kaggle"
    if Path("/content").is_dir():
        return "colab"
    return "local"


_rt = globals().get("RUNTIME")
if _rt is None or (isinstance(_rt, str) and not str(_rt).strip()):
    _rt = (
        os.environ.get("FOL_RUNTIME", "").strip().lower()
        or os.environ.get("LOGIC_RUNTIME", "").strip().lower()
        or None
    )
if _rt is None:
    RUNTIME = _fol_detect_runtime()
else:
    RUNTIME = str(_rt).strip().lower()
    if RUNTIME not in ("local", "colab", "kaggle"):
        raise ValueError("RUNTIME phải là 'local', 'colab', 'kaggle' hoặc None")

NEST = "Logic_Based_Educational_Queries_Project"

if RUNTIME == "local":
    print("RUNTIME=local — bỏ qua clone. Đảm bảo cwd hoặc LOGIC_PROJECT_ROOT trỏ tới", NEST)
    here = Path.cwd().resolve()
    found = None
    for base in (here, *here.parents):
        for cand in (base, base / NEST):
            if (cand / "src" / "services").is_dir():
                found = cand.resolve()
                break
        if found:
            break
    if found:
        os.environ["LOGIC_PROJECT_ROOT"] = str(found)
        print("LOGIC_PROJECT_ROOT =", os.environ["LOGIC_PROJECT_ROOT"])
    else:
        print("Chưa tự set được LOGIC_PROJECT_ROOT — Phase 3 sẽ tìm theo cwd / biến môi trường.")
else:
    # ===== GitHub — classic PAT; "" → GITHUB_TOKEN / Kaggle Secret
    TOKEN = ""  # repo private: dán PAT tạm thời — không commit token thật

    GITHUB_OWNER_REPO = "fishperson113/Exact_2026_Laplace-s_Red_Devils"
    GIT_BRANCH = "Son/Logic_Based_Educational_Queries"

    _t = TOKEN.strip()
    if not _t:
        _t = os.environ.get("GITHUB_TOKEN", "").strip()
    if not _t:
        try:
            from kaggle_secrets import UserSecretsClient

            _t = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            _t = ""

    if RUNTIME == "colab":
        CLONE_ROOT = Path("/content/project")
    else:
        CLONE_ROOT = Path("/kaggle/working/project")

    CLONE_ROOT.parent.mkdir(parents=True, exist_ok=True)

    if not (CLONE_ROOT / ".git").is_dir():
        if _t:
            url = f"https://{_t}@github.com/{GITHUB_OWNER_REPO}.git"
        else:
            url = f"https://github.com/{GITHUB_OWNER_REPO}.git"
        subprocess.run(["git", "clone", url, str(CLONE_ROOT)], check=True)
    else:
        print("Đã có clone:", CLONE_ROOT)

    subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--all"], check=False)
    subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", GIT_BRANCH], check=True)

    nest = CLONE_ROOT / NEST
    if not (nest / "src" / "services").is_dir() and (CLONE_ROOT / "src" / "services").is_dir():
        nest = CLONE_ROOT
    assert (nest / "src" / "services").is_dir(), f"Không thấy src/services: {nest}"

    os.environ["LOGIC_PROJECT_ROOT"] = str(nest.resolve())
    print("LOGIC_PROJECT_ROOT =", os.environ["LOGIC_PROJECT_ROOT"])
    print("Branch:", GIT_BRANCH)

print("RUNTIME =", RUNTIME)


RUNTIME=local — bỏ qua clone. Đảm bảo cwd hoặc LOGIC_PROJECT_ROOT trỏ tới Logic_Based_Educational_Queries_Project
LOGIC_PROJECT_ROOT = /home/Logic_Based_Educational_Queries_Project
RUNTIME = local


## Phase 2 — Token HF + đăng nhập Hub

- **Kaggle:** Secret `HF_Token` (ưu tiên) hoặc biến môi trường.
- **Colab / local:** `HF_TOKEN` / `HUGGINGFACE_HUB_TOKEN` hoặc file `.env` trong project.


In [35]:
import os
from pathlib import Path

HF_Token = os.environ.get("HF_TOKEN")
if Path("/kaggle").is_dir():
    try:
        from kaggle_secrets import UserSecretsClient

        HF_Token = UserSecretsClient().get_secret("HF_Token")
    except Exception:
        HF_Token = ""
if not HF_Token:
    HF_Token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""


In [36]:
import torch
import huggingface_hub

if HF_Token:
    try:
        huggingface_hub.login(token=HF_Token, add_to_git_credential=False)
    except Exception as e:
        print("HF login (tuỳ chọn):", e)

print("Thiết bị:", "cuda" if torch.cuda.is_available() else "cpu")


Thiết bị: cuda


## Phase 3 — Bootstrap


In [37]:
import os, sys
from pathlib import Path

NEST = "Logic_Based_Educational_Queries_Project"


def logic_repo_root() -> Path:
    for key in ("LOGIC_PROJECT_ROOT", "EXACT_PROJECT_ROOT"):
        v = os.environ.get(key, "").strip()
        if v:
            p = Path(v).expanduser().resolve()
            for cand in (p, p / NEST):
                if (cand / "src" / "services").is_dir():
                    return cand.resolve()
    kin = Path("/kaggle/input")
    if kin.is_dir():
        for top in sorted(kin.iterdir()):
            if not top.is_dir():
                continue
            for cand in (top, top / NEST):
                if (cand / "src" / "services").is_dir():
                    return cand.resolve()
            try:
                for sub in top.iterdir():
                    if not sub.is_dir():
                        continue
                    for cand in (sub, sub / NEST):
                        if (cand / "src" / "services").is_dir():
                            return cand.resolve()
            except OSError:
                pass
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base, base / NEST):
            if (cand / "src" / "services").is_dir():
                return cand.resolve()
    raise FileNotFoundError(
        "Không thấy src/services. local: mở notebook từ .../Logic_Based_Educational_Queries_Project/notebooks "
        "hoặc export LOGIC_PROJECT_ROOT. colab/kaggle: chạy ô clone (RUNTIME=colab|kaggle) hoặc Add Data; "
        "hoặc LOGIC_PROJECT_ROOT=..."
    )


R = logic_repo_root()
sys.path.insert(0, str(R / "src"))
from services.config import load_dotenv_for_logic

os.chdir(R / "notebooks")
print("PROJECT_ROOT:", R, "| cwd:", Path.cwd(), "| .env:", load_dotenv_for_logic())


PROJECT_ROOT: /home/Logic_Based_Educational_Queries_Project | cwd: /home/Logic_Based_Educational_Queries_Project/notebooks | .env: None


## Phase 4 — Cấu hình
`FOL_EVAL_FOL_MAX_SAMPLES=all` — greedy eval **sau train** trên đủ train/dev/test (lâu). Baseline test trong code **luôn full**.


In [38]:
import os
import sys
import warnings
from pathlib import Path


def _ensure_project_src() -> Path:
    """Tìm thư mục có src/services và chèn src vào sys.path (ô này có thể chạy độc lập sau kernel restart)."""
    for anchor in [Path.cwd(), *Path.cwd().parents]:
        src = anchor / "src"
        if (src / "services" / "config_fol.py").is_file():
            p = str(src.resolve())
            if p not in sys.path:
                sys.path.insert(0, p)
            return anchor.resolve()
    raise ModuleNotFoundError(
        "Không tìm thấy src/services (FOL). Hãy %cd tới Logic_Based_Educational_Queries_Project "
        "hoặc .../notebooks trong repo, hoặc chạy ô Phase 3 Bootstrap trước."
    )


_ensure_project_src()


def _fol_unsloth_import_ok() -> bool:
    try:
        import unsloth  # noqa: F401

        return True
    except Exception:
        return False


def _fol_sync_fol_use_unsloth_env() -> None:
    """Chỉ chỉnh env khi người dùng *đã* đặt FOL_USE_UNSLOTH=true mà không import được unsloth.

    Nếu biến chưa có: không gán gì — để ``FolSFTConfig.from_env()`` (configs/fol_model.yaml)
    quyết định; tránh ép ``FOL_USE_UNSLOTH=false`` làm mất ``use_unsloth: true`` từ YAML.
    """
    key = "FOL_USE_UNSLOTH"
    raw = os.environ.get(key, "").strip().lower()
    if not raw:
        return
    want = raw in ("1", "true", "yes", "y", "on")
    if want and not _fol_unsloth_import_ok():
        warnings.warn(
            f"{key}={os.environ.get(key)!r} nhưng không import được `unsloth` — chuyển sang false "
            "(train bằng transformers thường). Cài: pip install unsloth (torch đúng bản) hoặc đặt false trong .env.",
            UserWarning,
            stacklevel=1,
        )
        os.environ[key] = "false"


os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("FOL_EVAL_FOL_MAX_SAMPLES", "all")
# Nếu đã export FOL_USE_UNSLOTH=true mà không import được unsloth → ô trên cảnh báo và ép false.
# Không set FOL_USE_UNSLOTH → để from_env() + configs/fol_model.yaml quyết định use_unsloth.
_fol_sync_fol_use_unsloth_env()
os.environ.setdefault("FOL_USE_ADAMW_8BIT", "true")
# Unsloth: NF4 (~2GB base) chỉ khi thiếu VRAM. GPU ~24GB: false → nạp base FP16 (load_in_16bit).
os.environ.setdefault("FOL_UNSLOTH_NF4", "false")

from services.config_fol import FolSFTConfig

cfg = FolSFTConfig.from_env()
print("Config OK", cfg.model_name)
print("HF repo:", cfg.resolved_hf_repo())
print("FOL_EVAL_FOL_MAX_SAMPLES:", cfg.eval_fol_max_samples)
print("use_unsloth:", cfg.use_unsloth, "| use_adamw_8bit:", cfg.use_adamw_8bit, "| responses_only:", cfg.unsloth_train_on_responses_only)
print("unsloth_load_in_4bit (NF4):", cfg.unsloth_load_in_4bit, "| eval_accumulation_steps:", cfg.eval_accumulation_steps, "| max_seq_length:", cfg.max_seq_length)


Config OK Qwen/Qwen2.5-3B-Instruct
HF repo: Laplaces-Red-Devils/fol-v01-origin-qwen2.5-3
FOL_EVAL_FOL_MAX_SAMPLES: None
use_unsloth: True | use_adamw_8bit: True | responses_only: True
unsloth_load_in_4bit (NF4): False | eval_accumulation_steps: 4 | max_seq_length: 2048


## Phase 4b — (Tuỳ chọn) Lọc FOL → thư mục con

Mặc định **logic + FOL** dùng chung `data/processed/train.csv`. Nếu muốn bản **chỉ** các dòng `len(premises_nl)==len(premises_fol)` sang thư mục riêng:

```python
from pathlib import Path
from data.fol_dataset import export_filtered_fol_csvs
root = Path(cfg.app_dir).resolve()
export_filtered_fol_csvs(root / "data/processed", root / "data/processed/fol_filtered")
```

Sau đó có thể đặt `FOL_SFT_PROCESSED_DIR` trỏ tới `fol_filtered` nếu chỉ train trên tập đã lọc.


## Phase 5 — Dataset + xem vài `eval_prompt` (train)


In [39]:
from models.fol_model.train import load_tokenizer
import data.fol_dataset as _fol_mod  # đường dẫn file — Kaggle: phải là .../src/data/fol_dataset.py trong repo
from data.fol_dataset import build_fol_dataset_dict
from models.fol_model.fol_preflight import print_fol_prompt_previews

print("data.fol_dataset:", _fol_mod.__file__)

tok = load_tokenizer(cfg)
dataset_dict, filt_stats = build_fol_dataset_dict(cfg, tok)
print("filter (dropped per split):", filt_stats)
print_fol_prompt_previews(dataset_dict, tok, n=3)


data.fol_dataset: /home/Logic_Based_Educational_Queries_Project/src/data/fol_dataset.py


Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

filter (dropped per split): {'train': 15, 'dev': 4, 'test': 2}

=== Xem trước 3 eval_prompt (train) ===

Ghi chú: các dòng đầu train thường trùng eval_prompt (cùng premises NL, khác câu hỏi). Đang hiển thị các prompt **khác nhau**; đặt unique_eval_prompt=False để xem đúng n dòng đầu.

--- mẫu 0 (index train=0, record_id=1 | q_idx=0) ---
<|im_start|>system
### Instruction
You are a First-Order Logic (FOL) translator. Convert each numbered natural-language (NL) premise into a precise FOL formula.

### Rules
1. Predicate names MUST be derived EXACTLY from the NL text — use full CamelCase (e.g. "UserFriendly", "CompatibleWithEcosystem"). NEVER abbreviate to single letters (e.g. U, C, E are FORBIDDEN unless the NL itself uses that letter).
2. Map each NL premise to exactly one FOL formula, in the same order and same count.
3. Output ONLY a JSON object with key "premises_fol" whose value is a list of strings.
4. No markdown fences, no explanation, no commentary outside the JSON.
5. NEVER int

## Phase 6 — Baseline (trước SFT)
**10** infer greedy ngẫu nhiên trên **train** + **full test** exact-match & NLL → `fol_preflight_baseline.json`.


In [40]:
'''
import gc
import torch
from models.fol_model.fol_preflight import run_preflight_baseline_and_save

_ = run_preflight_baseline_and_save(
     cfg,
     tok,
     dataset_dict,
     random_infer_n=10,
     random_seed=42,
     preview_rows=0,
 )
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''

'\nimport gc\nimport torch\nfrom models.fol_model.fol_preflight import run_preflight_baseline_and_save\n\n_ = run_preflight_baseline_and_save(\n     cfg,\n     tok,\n     dataset_dict,\n     random_infer_n=10,\n     random_seed=42,\n     preview_rows=0,\n )\ngc.collect()\nif torch.cuda.is_available():\n    torch.cuda.empty_cache()\n'

## Phase 7 — Supervised fine-tune (LoRA)


In [49]:
import gc
import torch
from models.fol_model.train import run_train_and_eval

train_out = run_train_and_eval(cfg)
print("Train OK:", train_out is not None)
if train_out is not None:
    _, _trainer, _tok2, _ds = train_out
    del _trainer, _tok2, _ds, train_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


[FOL train] Bắt đầu: tokenizer + dataset + model (bước này có thể vài phút, chưa có thanh epoch).
[FOL train][Unsloth] Nạp model + tokenizer (gradient checkpointing kiểu unsloth nếu bật).
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
[FOL train][Unsloth] Trainer mixed precision: bf16=True, fp16=False (dtype tham số: torch.bfloat16).
[FOL train] Đã nạp tokenizer (Unsloth).


Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

...

Filter (num_proc=48):   0%|          | 0/77 [00:00<?, ? examples/s]

[FOL train] Bắt đầu trainer.train() …


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,0.147267,0.150195
2,0.075058,0.138218
3,0.028705,0.188271
4,0.016085,0.207000
5,0.008969,0.226396
6,0.003919,0.233291
7,0.001832,0.251726


KeyboardInterrupt: 

## Phase 8 — Push Hub
`FOL_PUSH_TO_HUB=true` hoặc `cfg.push_to_hub = True`.


In [ ]:
import os
from services.hub_push import push_merged_fol_lora

if not cfg.push_to_hub:
    print("push_to_hub=False — bỏ qua.")
else:
    token = (
        HF_Token
        or os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        or ""
    ).strip()
    if not token:
        raise ValueError("Thiếu token HF.")
    url = push_merged_fol_lora(cfg, token)
    print("Pushed:", url)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/798 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Phase 9 — Hub: metrics + **20** mẫu test ngẫu nhiên
Cần đã push (Phase 8).


In [48]:
import gc
import os
import torch
from models.fol_model.hub_reload_eval import (
    evaluate_fol_hub_model,
    print_fol_hub_eval_summary,
)

HUB_REPO = cfg.resolved_hf_repo()
tok3 = (
    HF_Token
    or os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    or ""
).strip() or None

res = evaluate_fol_hub_model(
    cfg,
    HUB_REPO,
    hf_token=tok3,
    greedy_eval_max_samples=20,
    random_test_infer_n=20,
    random_test_seed=7,
)
print_fol_hub_eval_summary(res, max_sample_print=5)
del res
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

[FOL greedy eval] train: 630 mẫu, batch_size=2


KeyboardInterrupt: 